# Qwen3.5-4B Fashion Agent LoRA Pilot

현재 확보된 Gold Data 80개만으로 데이터 변환, Base Model 평가, LoRA 학습, 학습 후 평가까지 전체 파이프라인을 확인합니다.

- Train / Validation / Test: 64 / 8 / 8
- 모델: `unsloth/Qwen3.5-4B`
- 방식: 16-bit LoRA (`load_in_4bit=False`, `load_in_16bit=True`)
- 목적: 최종 성능 확보가 아닌 파이프라인 검증
- 출력 규칙: 현행 데이터와 서비스 설계에 맞춰 3 LOOK 유지

> 합성 데이터 500~1,000개와 HITL 검수가 완료되기 전에는 이 결과를 최종 모델 성능으로 해석하지 않습니다.


In [ ]:
# Colab에서 한 번만 실행합니다.
!pip install -qU unsloth datasets trl accelerate bitsandbytes


## 1. Pilot 데이터 업로드

함께 제공된 `qwen35_pilot_data.zip`을 업로드합니다.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.endswith(".zip")), None)
assert zip_name is not None, "qwen35_pilot_data.zip 파일을 업로드하세요."

data_root = Path("/content/qwen35_pilot_data")
data_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(data_root)

for required in ["train.jsonl", "validation.jsonl", "test.jsonl", "tool_definitions.json"]:
    assert (data_root / required).exists(), f"누락된 파일: {required}"

print("데이터 준비 완료:", sorted(path.name for path in data_root.iterdir()))


## 2. Dataset과 모델 설정

In [ ]:
import json
import os
import re
from pathlib import Path

import torch
from datasets import load_dataset

SEED = 3407
MAX_SEQ_LENGTH = 4096
MODEL_NAME = "unsloth/Qwen3.5-4B"
OUTPUT_DIR = "/content/qwen35-fashion-pilot"

dataset = load_dataset(
    "json",
    data_files={
        "train": str(data_root / "train.jsonl"),
        "validation": str(data_root / "validation.jsonl"),
        "test": str(data_root / "test.jsonl"),
    },
)
TOOLS = json.loads((data_root / "tool_definitions.json").read_text(encoding="utf-8"))
print(dataset)


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print("model dtype:", next(model.parameters()).dtype)
print("model device:", next(model.parameters()).device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))


## 3. Qwen Chat Template 적용 및 길이 검사

In [ ]:
def render_messages(example, add_generation_prompt=False, prompt_only=False):
    messages = json.loads(example["messages_json"])
    if prompt_only:
        first_assistant = next(
            (index for index, message in enumerate(messages) if message["role"] == "assistant"),
            len(messages),
        )
        messages = messages[:first_assistant]

    kwargs = {
        "tokenize": False,
        "add_generation_prompt": add_generation_prompt,
        "enable_thinking": False,
    }
    if example["type"] == "tool_calling":
        kwargs["tools"] = TOOLS
    return tokenizer.apply_chat_template(messages, **kwargs)


def add_text(example):
    return {"text": render_messages(example)}


formatted = dataset.map(add_text)

def token_length(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

for split_name in ["train", "validation", "test"]:
    lengths = [token_length(text) for text in formatted[split_name]["text"]]
    print(split_name, {"count": len(lengths), "max": max(lengths), "mean": round(sum(lengths) / len(lengths), 1)})
    assert max(lengths) <= MAX_SEQ_LENGTH, f"{split_name}에 {MAX_SEQ_LENGTH} 토큰 초과 데이터가 있습니다."

print(formatted["train"][0]["text"][:1200])


## 4. Base Model 평가

Test 8개를 LoRA 적용 전에 실행해 Baseline 응답을 저장합니다. 처음 실행은 커널 초기화로 시간이 걸릴 수 있습니다.


In [ ]:
def generate_for_test(current_model, rows, max_new_tokens=1024):
    FastLanguageModel.for_inference(current_model)
    results = []
    for row in rows:
        prompt = render_messages(row, add_generation_prompt=True, prompt_only=True)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to("cuda")
        with torch.inference_mode():
            output = current_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
            )
        generated = output[:, inputs["input_ids"].shape[1]:]
        response = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
        results.append(
            {
                "id": row["id"],
                "type": row["type"],
                "expected_first_action": row["expected_first_action"],
                "response": response,
            }
        )
        print(row["id"], "완료")
    return results


test_rows = [dataset["test"][i] for i in range(len(dataset["test"]))]
baseline_results = generate_for_test(model, test_rows)
Path("/content/baseline_results.json").write_text(
    json.dumps(baseline_results, ensure_ascii=False, indent=2), encoding="utf-8"
)


## 5. 16-bit LoRA Pilot 학습

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

model.print_trainable_parameters()


In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    packing=False,
    dataset_num_proc=1,
    report_to="none",
    seed=SEED,
)

trainer_kwargs = dict(
    model=model,
    train_dataset=formatted["train"],
    eval_dataset=formatted["validation"],
    args=training_args,
)

try:
    trainer = SFTTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer


In [ ]:
train_result = trainer.train()
print(train_result.metrics)

adapter_dir = "/content/qwen35-fashion-pilot-lora"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("LoRA 저장 완료:", adapter_dir)


## 6. 학습 후 동일 Test 평가

In [ ]:
finetuned_results = generate_for_test(model, test_rows)
Path("/content/finetuned_results.json").write_text(
    json.dumps(finetuned_results, ensure_ascii=False, indent=2), encoding="utf-8"
)


In [ ]:
def extract_json(text):
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def score_result(item):
    expected = item["expected_first_action"]
    response = item["response"]
    if item["type"] == "fashion_recommendation":
        parsed = extract_json(response)
        looks = parsed.get("looks") if isinstance(parsed, dict) else None
        return {
            "json_valid": parsed is not None,
            "three_looks": isinstance(looks, list) and len(looks) == 3,
            "first_action_correct": None,
        }
    if expected == "direct_answer":
        used_tool = any(name in response for name in [
            "get_user_context", "get_wardrobe", "get_weather", "get_calendar_events"
        ])
        return {"json_valid": extract_json(response) is not None, "three_looks": None, "first_action_correct": not used_tool}
    return {"json_valid": None, "three_looks": None, "first_action_correct": expected in response}


comparison = []
for before, after in zip(baseline_results, finetuned_results):
    comparison.append(
        {
            "id": before["id"],
            "type": before["type"],
            "expected_first_action": before["expected_first_action"],
            "baseline_score": score_result(before),
            "finetuned_score": score_result(after),
            "baseline_response": before["response"],
            "finetuned_response": after["response"],
        }
    )

Path("/content/pilot_comparison.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8"
)

for item in comparison:
    print(item["id"], "base=", item["baseline_score"], "fine-tuned=", item["finetuned_score"])


## 7. 결과 파일 다운로드

In [ ]:
import shutil
from google.colab import files

result_root = Path("/content/pilot_result")
result_root.mkdir(exist_ok=True)
for file_name in ["baseline_results.json", "finetuned_results.json", "pilot_comparison.json"]:
    shutil.copy2(Path("/content") / file_name, result_root / file_name)
shutil.copytree(adapter_dir, result_root / "qwen35-fashion-pilot-lora", dirs_exist_ok=True)

archive = shutil.make_archive("/content/qwen35_fashion_pilot_result", "zip", result_root)
print("결과 압축 완료:", archive)
files.download(archive)


## 완료 판단

다음 항목이 모두 확인되면 WBS 3.6의 **Pilot Fine-Tuning** 실행은 완료입니다.

- Train loss가 기록되고 학습이 오류 없이 종료됨
- Validation loss가 산출됨
- LoRA adapter가 저장됨
- Base/Fine-tuned Test 8개 응답이 모두 생성됨
- `pilot_comparison.json`에서 JSON 형식과 Tool 선택 변화를 확인함

최종 학습은 합성 데이터와 HITL 정제가 끝난 뒤 동일한 Notebook에서 데이터 파일만 교체하여 진행합니다.
